### Instructions
- This notebook reproduces the images shown in the preprint [Saint Fleur et al (2025)](https://doi.org/10.5194/egusphere-2025-4244). It is a clik and go, but we encourage you to check if it runs on your system first. You may need to have downloaded the related data_paper first, then jump to the *TODO* flags in the cells below and get them adapted properly.

- To run all in one shot, once ready, click on the **Run All** or **>>** button above
- The images will be saved in a sub-folder of the data_paper, unless you change it, their name was made meaningful for easier catching-up


[Note] : The structure of this notebook is not optimized, since most of the scripts could be simplified. We provide it in its current state as a way to minimize surprises.

In [ ]:
from glob import glob
import os
from typing import Any
from statsmodels.tsa import stattools
from matplotlib.ticker import MultipleLocator
import numpy as np
import pandas as pd
import pickle
import scipy
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.lines import Line2D
from pathlib import Path
import warnings
from post_process_v2.pp_utils import read_basin_list_from_file
from tqdm import tqdm
from datetime import datetime
import matplotlib.patches as mpatches
from post_process_v2.pp_utils import make_rank_for_basin
from sklearn.metrics import mean_absolute_error
warnings.filterwarnings("ignore")

---
# General options
---

In [ ]:
PATH_TO_DATA_PAPER= r"Y:\repo_egu24"    #TODO 1: adapt to your reality
CASE_CT = CASE_CAMELS = "fr"            #TODO 2: switch between "fr" and "us" aka CAMELS-US and CAMELS-FR
SAVE_IMAGES = False                     #TODO 3: check to save images

is_fr = "fr" in CASE_CT
GENERAL = {
            "context_list": {"fr": ["lstm", "no_in_mlp"],
                            "us": ["sacsma", "lstm", "no_in_mlp", "sma_in_mlp",
                                   "lstm_in_mlp", "predict_sma_error", "predict_lstm_error"]},
            
            "context_pair": {"fr": [("lstm", "no_in_mlp")],
                      "us": [("sacsma", "no_in_mlp", "sma_in_mlp", "predict_sma_error"),
                             ("lstm", "no_in_mlp", "lstm_in_mlp", "predict_lstm_error")]},
            
            "context_dict": {"us": {"lstm": ["lstm", "no_in_mlp", "lstm_in_mlp", "predict_lstm_error", "natural_records"],
                                "sacsma": ["sacsma", "no_in_mlp", "sma_in_mlp", "predict_sma_error", "natural_records"]},
                           "fr": {"lstm": ["lstm", "no_in_mlp", "natural_records"],
                                  "sacsma":[]},},
            
            "sep_box_context": {"us": {"lstm": ("no_in_mlp", "lstm_in_mlp", "predict_lstm_error", "persistent", "natural_records"),
                            "sacsma":("no_in_mlp", "sma_in_mlp", "predict_sma_error", "persistent", "natural_records")},
            
                     "fr": {"lstm": ("no_in_mlp", "persistent", "natural_records"),
                            "sacsma":()}},
            
            "forecast_cases":{"fr": ["perf", "clim", "hind", "real"],
                            "us": ["perf", "clim", "hind"]},
            
            "period": {"fr": ("20180901", "20210831"), 
                     "us": ("19891001", "19910930")},
            
            "data_root":fr"{PATH_TO_DATA_PAPER}\data_paper\data",
            "processed_root": fr"{PATH_TO_DATA_PAPER}\data_paper\processed",
            "metrics_root": fr"{PATH_TO_DATA_PAPER}\data_paper\metrics_grouped",    #TODO 5: Adapt to your reality
            "images_root": fr"{PATH_TO_DATA_PAPER}\data_paper\images_temp",         #TODO 6: Adapt to your reality
          }
generic_style = {
        "no_in_mlp": {"short":"MLP", "short_0": "MLP", "short_1": "MLP Simple", "ref": "MLP", "ls": "-", "marker": "s",
               "marker_rel": "s", "marker_id": "s","color_id": "teal", "color": "teal", "color2": "teal", "ms": 8},
        "sma_in_mlp": {"short":"$i$-SAC", "short_0": "$i$-SACSMA", "short_1": "MLP Informed by SACSMA", "ref": "MLP Informed",
                "ls": "--", "marker": "o", "marker_rel": "o", "marker_id": "o","color_id": "darkorange",
                "color": "magenta", "color2": "darkorange","ms": 7},
        "lstm_in_mlp": {"short":"$i$-LSTM","short_0": "$i$-LSTM", "short_1": "MLP Informed by LSTM", "ref": "MLP Informed",
                 "ls": "--", "marker": "^", "marker_rel": "o", "marker_id": "^", "color_id": "darkorange",
                 "color": "darkorange", "color2": "darkorange", "ms": 7},
        "predict_sma_error": {"short": "$\epsilon$SAC", "short_0": "$\\epsilon$SACSMA", "short_1": "$\\epsilon$SAC-SMA post-processed",
                       "ref": "Error-Postprocessing", "ls": "-.", "marker": "P", "marker_rel": "^", "marker_id": "P",
                       "color_id": "darkorange","color": "magenta",
                       "color2": "gold","ms": 8},
        "predict_lstm_error": {"short": "$\epsilon$LSTM","short_0": "$\\epsilon$LSTM", "short_1": "$\\epsilon$LSTM post-processed",
                        "ref": "Error-Postprocessing", "ls": "-.", "marker": "*", "marker_rel": "^", "marker_id": "*", 
                        "color_id": "darkorange","color": "darkorange",
                        "color2": "gold", "ms": 10},
        "sacsma": {"short": "SACSMA_AN17", "short_0": "SACSMA", "short_1": "SACSMA_Newman2017", "ref": "Initial Model", "ls": ":",
            "marker": "", "marker_rel": "P", "marker_id": ".","color_id": "red", "color": "blue", "color2": "red", "ms": 5},
        "lstm": {"short": "LSTM_FK19","short_0": "LSTM", "short_1": "LSTM_Kratzert2019", "ref": "Initial Model", "ls": ":",
          "marker": "", "marker_rel": "P", "marker_id": ".","color_id": "red", "color": "red","color2": "red",  "ms": 5},
        "persistent": {"short": "Persistent Model","short_0": "Persistent Model", "short_1": "Persistent Model", "ref": "Persistent Model", "ls": "-",
                "marker": "", "marker_rel": "", "marker_id": "", "color_id": "gray", "color": "silver", "color2": "silver","ms": 1},
        "natural_records": {"short": "Natural_Records","short_0": "Past Observations", "short_1": "Past Observations", "ref": "Past Observations",
                     "ls": "-", "marker": "", "marker_rel": "x", "marker_id": "", "color_id": "k", "color": "dimgray", "color2": "dimgray","ms": 1},
        }

## Dependant options

In [ ]:
DATA_DIR = GENERAL.get("data_root") + f"/camels{CASE_CAMELS}"
BASE_PP = GENERAL.get("processed_root") + f"/{CASE_CAMELS}"
DATA_ROOT = GENERAL.get("data_root") + f"/camels{CASE_CAMELS}"
RAW_FMT_DATA_PATH = DATA_ROOT + "/all_sim_obs_lstm"
METRICS_MAIN_DIR = GENERAL.get("metrics_root") + f"/{CASE_CAMELS}" 
IMAGES_BOX = GENERAL.get("images_root") + f"/{CASE_CAMELS}"
os.makedirs(IMAGES_BOX, exist_ok=True)

list_basin_531 = read_basin_list_from_file(DATA_DIR + "/basins_list")
list_basin_56 = read_basin_list_from_file(DATA_DIR +"/basins_56")
list_basin_531.sort()
list_basin_56.sort()

FORCAST_CASES = GENERAL.get("forecast_cases").get(CASE_CAMELS)
list_ctx = GENERAL.get("context_list").get(CASE_CAMELS)
ctx_pair_list = GENERAL.get("context_pair").get(CASE_CAMELS)
period = GENERAL.get("period").get(CASE_CAMELS)
cx_list_dict = GENERAL.get("context_dict").get(CASE_CAMELS)
sep_bxp =  GENERAL.get("sep_box_context").get(CASE_CAMELS)

FCST_DICT = {"clim":"Climatology", "perf":"perfect", "hind":"Hindcast", "real":"Forecast Archives"}
list_hp = [1, 2, 3, 4, 5, 6, 7]
list_hp3 = [1, 3, 7]

ctx_name_dict0 = {k:v["short_0"] for k,v in generic_style.items()}
fliers_props = dict(marker='.', markerfacecolor='k', markersize=3, linestyle='none', markeredgecolor=None)

USED_QT = [0.10, 0.25, 0.50, 0.75, 0.9]
DROUGHT_QT = [0.01, 0.05, 0.10, 0.25, 0.5]
FLOOD_QT = [0.75, 0.90, 0.95, 0.99]

In [ ]:
sns.set_style("ticks")
def custom_legend(axe: Any, dict_labels: dict) -> Any:
    """Reformat a plot legend according a dict"""
    line_ = axe.get_legend_handles_labels()[0]
    out_label=[dict_labels[a] for a in axe.get_legend_handles_labels()[1]]
    return line_, out_label

def read_pqt(f):
    return pd.read_parquet(f, engine="pyarrow")

def set_step_locator(x:float=0.5):
    """Set step on axis"""
    return mtick.MultipleLocator(x)

def get_xcor(act:pd.Series, react:pd.Series):
    """Make cross correlation"""
    return stattools.ccf(act, react)

def make_ecdf(vector: np.ndarray):
    """make ECDF data"""
    bin_ = np.sort(vector)
    cdf_ = np.arange(1, len(bin_)+1)/float(len(bin_))
    return bin_, cdf_

sns.set_style('ticks')

---
# A . Short analysis on data
---

### A.1. Basins selection

In [ ]:
if "us" in CASE_CT:
    lstm_nse = pickle.load(open("../all_metrics.p", "rb"))["NSE"]["lstm_NSE"]
    lstm_nse = pd.DataFrame().from_dict(lstm_nse, orient="columns")[["ensemble"]]
    selected_bv = list_basin_56
    size_bv=len(selected_bv)
    fig, ax = plt.subplots(figsize=(8, 4))
    
    ax = sns.ecdfplot(lstm_nse["ensemble"], label="All basins", ax=ax, lw=0.1, marker=".", color="k", ms=6)
    ax = sns.ecdfplot(lstm_nse.loc[lstm_nse.index.isin(selected_bv), "ensemble"], 
                      label="Sampled basins", ax=ax, marker="o", ms=8, lw=0., 
                      color="orange", alpha=0.8,
                      markeredgecolor="darkorange")
    ax.legend(fontsize=12)
    ax.set_title(f"Sampled basins (Nb={size_bv})", fontsize=13)
    ax.set_xlabel("NSE (Kratzert et al., 2019)", fontsize=12)
    ax.set_ylabel("Proportion", fontsize=12)
    ax.set_xlim((0., 1.))
    ax.set_ylim((0., 1.))
    ax.tick_params(which="major", axis="both", labelsize=12)
    ax.grid()
    if SAVE_IMAGES:
        plt.savefig(f"{IMAGES_BOX}/selected_{size_bv}.png", bbox_inches="tight", dpi=300)

### A.2. Cross correlation

In [ ]:
all_q_r = []
for i, e_tg in enumerate(["P_CM", "Q_Obs"]):
    all_c = pd.DataFrame(index=np.arange(200))
    for bvx in tqdm(list_basin_531):
        data = pd.read_csv(RAW_FMT_DATA_PATH + f"/{bvx}.txt", sep=";", 
                           parse_dates=True, index_col="Date")
        all_c[bvx] = get_xcor(data.Q_Obs, data[e_tg])[:200]
    all_c["feature"] = ["R-Q cross", "Q-Q auto"][i]
    all_q_r.append(all_c)
all_q_r = pd.concat(all_q_r, axis = 0)
all_q_r.index.name = "Lag"
all_q_r = all_q_r.set_index("feature", append=True).reorder_levels(["feature", "Lag"]).fillna(-0.005)
all_q_ = all_q_r.quantile([0.1, 0.5, 0.9], axis=1).T
all_q_.columns = ["Q10", "Q50", "Q90"]
all_q_[all_q_<0.] = -0.005

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3), sharey=True)
for i, d_cx in enumerate(["R-Q cross", "Q-Q auto"]):
    ax=axs[i]
    qq_x = d_cx.split(' ')[0]+[" Cross-Correlogram", " Auto-Correlogram"][i]
    qq_p = all_q_r.xs(d_cx, level="feature")
    _= ax.plot(qq_p, label=None, lw=0.2, c="dimgray")
    _= ax.plot(qq_p[qq_p.columns[-1:]], label=f"Basins {qq_x}", lw=0.2, c="dimgray")
    _= ax.plot(qq_p.median(axis=1), marker=".", ms=4, label=f"Median {qq_x}", lw=.8, c="k")
    x,y = [30, 10][i], [0.03, 0.2][i]
    x_fr,y_fr = [70, 50][i], [0.03, 0.2][i]
    ax.plot(x, y, marker="o", ms=10, mfc="orange", mec="k", lw=2)
    ax.set(ylim=(0, 1.02), xlim=(-1, 80), xlabel="Lag (days)", ylabel="Correlation Score" if i==0 else "")
    ax.grid(ls="--", lw=0.2)
    ax.axvline(x, ls="-.", lw=0.5, color="k")
    ax.axhline(y, ls="-.", lw=0.5, color="k")
    if CASE_CT =="fr":
        ax.plot(x_fr, y_fr, marker="o", ms=10, mfc="teal", mec="k", lw=2)
        ax.axvline(x_fr, ls="-.", lw=0.5, color="k")
        ax.axhline(y_fr, ls="-.", lw=0.5, color="k")
    ax.legend()
plt.subplots_adjust(wspace=0.1)
if SAVE_IMAGES:
    fig.savefig(IMAGES_BOX+f"/Cross_corr_v2_{CASE_CT}_L.png", bbox_inches="tight", dpi=300)

### A.3 Rank diagram of RAINFALL and PET

In [ ]:
feats_list = ["P_CM", "ETP_CM"]
dict_name = {"P_CM": "Precipitation", "ETP_CM":"Potential Evapotranspiration (PET)"}
dict_rk = []
for basin in tqdm(list_basin_56):
    df_x = pd.read_csv(RAW_FMT_DATA_PATH +  f"/{basin}.txt", sep=";", index_col="Date", parse_dates=True)
    rk, mb_x = make_rank_for_basin(df_0=df_x, period=period, feats=feats_list)
    rk["basin"] = basin
    rk[rk.isna()] = 0.
    dict_rk.append(rk)
df_rk = pd.concat(dict_rk, axis=0)
df_rk = df_rk.set_index("basin", append=True).sort_index()


n_rk = len(df_rk.index.get_level_values("Rank").unique())
fg, axs = plt.subplots(1, 2, figsize=(9, 3), sharex="all", sharey="all")
for i, feat in enumerate(feats_list):
    ax = axs[i]
    sns.barplot(df_rk[[feat]].melt(ignore_index=False), x="Rank", y="value", ax=ax, 
                palette=["teal"],capsize=0.2, err_kws=dict(lw=0.8, c="k"))
    ax.axhline(1/n_rk, ls="--", lw=0.8, color="grey")
    ax.set_title(dict_name[feat], y=0.85, fontsize=14)
    ax.set(ylim=(0., 0.06 if "fr" in CASE_CT else 0.1), ylabel="Frequency", xlabel="Ranks")
    labels = [str(i) if i%2!=0 else "" for i in df_rk.index.get_level_values('Rank').unique()]
    ax.set_xticklabels(labels)
    ax.tick_params(labelsize=11)
sns.despine()
plt.tight_layout()
if SAVE_IMAGES:
    plt.savefig(f"{IMAGES_BOX}/Climatology_variability_for_P_n_PET_{CASE_CT}.png", bbox_inches="tight")

---
# B. Perfect forecast post-processing
---

In [ ]:
obs_naive = pd.read_parquet(BASE_PP + "/obs_naive.parquet.gzip", engine="pyarrow")
if is_fr:
    obs_naive = obs_naive.swaplevel(0,1)

## B.1 Regression metrics

In [ ]:
pp_perf_path = rf"{METRICS_MAIN_DIR}/postp_perf"
res_pf = pd.read_parquet(os.path.join(pp_perf_path, "pp_metrics_pandas.parquet.gzip"), engine="pyarrow")
res_pf.head()

### B.1 (bis) Benchmarking NSE

In [ ]:
## Load from Kratzert et al (2019)
BM_FK19 = pickle.load(open(r"../all_metrics.p", "rb"))
BM_CM_US = pd.concat([pd.Series(BM_FK19["NSE"]["benchmarks"]["SAC_SMA"]).to_frame("SACSMA"),
                      pd.Series(BM_FK19["NSE"]["lstm_MSE"]["ensemble"]).to_frame("LSTM")], axis=1)

## Load from Hashemi et al. (2022) and local GR4J runs from Otis
RH_22 = pd.read_csv(r"../nse_gr4j_lstm_Hashemi_2022.txt", sep=";", index_col="number")
NSE_GR_UGE = pd.read_csv(r"../nse_gr4j_otis_uge.txt", sep=";", index_col="basin")

## Load from this study in France
NSE_FR = read_pqt(rf'{GENERAL.get("metrics_root")}\fr\postp_perf\pp_metrics_pandas.parquet.gzip')
NSE_LSTM_UGE = NSE_FR["nse_on_median"].xs("lstm", level="context").to_frame("LSTM").xs(1, level="hp")
NSE_GR_LSTM_UGE_26 = pd.concat([NSE_GR_UGE, NSE_LSTM_UGE], axis=1).sort_index()

MLP_BS = NSE_FR["nse_on_median"].xs("no_in_mlp", level="context").unstack("hp")[[1,3,7]].add_prefix("hp")
MLP_BS.columns = [f"MLP\n ({c})" for c in MLP_BS.columns]

In [ ]:
fig, axs=plt.subplots(1, 4, figsize=(10, 3), constrained_layout=True, sharey=True)
for i, d_sc in enumerate([BM_CM_US, RH_22, NSE_GR_LSTM_UGE_26, MLP_BS]):
    ax=axs[i]
    palette_ = ["b","darkorange"] if i<3 else ["teal"]*3
    sns.boxplot(d_sc,
                ax=ax, palette=palette_, width=0.6, fliersize=2, linewidth=1.2, linecolor="k")
    ax.set(ylim=(0, 1.02), title=["Kratzert et al. (2019)", "Hashemi et al. (2022)", "Present study", "MLP with CAMELS-FR \n(Present Study)"][i],)
    ax.grid()
    ax.set_ylabel("Nash & Sutcliffe (NSE)", fontsize=12)
    ax.set_xlabel(["Models with CAMELS-US", "Models (basins in France)", "Models with CAMELS-FR", ''][i], fontsize=12)
    ax.tick_params(labelsize=12)
if SAVE_IMAGES:
    plt.savefig(GENERAL.get("images_root") +"/NSE_LSTM_vs_GR4J_n_SACSMA_MLP_v2c.png", dpi=300, bbox_inches="tight")

### B.1.1 Persistence, NSE, KGE

In [ ]:
def bi_boxplot_v0(res_df, crit_, bm_name, y_lim=(0.,1.02), join_persistent:bool=False):
    """
    Plot a bi-box with benchmark on the left-most-single, un-fit for pers-type score
    
    :param
        - res_df: multiindex dataframe of scores
        - crit_: the criterion name
        - bm_name: which of lstm or sacsma
        - y_lim: pass the limits of the Y-axis
    """
    fig_x, (ax_b, ax_d) = plt.subplots(1, 2, figsize=(8, 3), gridspec_kw={'width_ratios': [1, 7]}, sharey=True)
    bm_d = res_df.xs(bm_name, level="context").xs(1, level="hp")[crit_].to_frame("0")
    da_d = res_df[res_df.index.get_level_values("context").isin(sep_bxp[bm_name])][[crit_]].reset_index("context")
    da_d["context"] = da_d["context"].map(ctx_name_dict0)
    da_d = da_d.set_index("context", append=True)
    sns.boxplot(bm_d, ax=ax_b, color="r", width=0.3, fliersize=0)
    hue_ord = [ctx_name_dict0[a] for a in sep_bxp[bm_name]]
    sns.boxplot(da_d, x="hp", y=crit_, hue="context", hue_order=hue_ord if join_persistent is True else hue_ord[:-2],
                ax=ax_d, palette=["teal", "darkorange", "gold", "grey"], fliersize=0)
    ax_b.set(ylim=y_lim, ylabel=crit_.split("_")[0].upper())
    ax_d.legend(loc="lower center", ncols=5, facecolor="lightgrey", fontsize=13)
    for i, xa in enumerate([ax_b, ax_d]):
        xa.axhline(float(bm_d.median()), ls="--", c="k", lw=0.6)
        xa.grid()
        xa.yaxis.set_major_locator(mtick.MultipleLocator(0.2))
        xa.tick_params(labelsize=12)
        xa.set_xlabel("Lead times (days)" if i==1 else bm_name.upper(), fontsize=14)
    plt.subplots_adjust(wspace=0.05)
    plt.savefig("_ttt.png", bbox_inches="tight", dpi=300)
    plt.close(fig_x)
    return fig_x

def mono_boxplot_v0(res_df, crit_, bm_name, y_lim=(0.,1.02), join_persistent:bool=False,
                    fig_s=(7, 3), n_col_lgd=4, y_locator:float=0.25):
    """
    Plot a bi-box with benchmark on the left-most-single, un-fit for pers-type score
    
    :param
        - res_df: multiindex dataframe of scores
        - crit_: the criterion name
        - bm_name: which of lstm or sacsma
        - y_lim: pass the limits of the Y-axis
    """
    fig_x, ax = plt.subplots(figsize=fig_s)
    bm_case = (bm_name,) + sep_bxp[bm_name]
    da_d = res_df[res_df.index.get_level_values("context").isin(bm_case)][[crit_]].reset_index("context")
    da_d["context"] = da_d["context"].map(ctx_name_dict0)
    da_d = da_d.set_index("context", append=True)
    hue_ord = [ctx_name_dict0[a] for a in bm_case]
    hue_ord = hue_ord if join_persistent is True else hue_ord[:-2]
    sns.boxplot(da_d, x="hp", y=crit_, hue="context", hue_order=hue_ord,
                ax=ax, 
                palette=["r", "teal", "silver", "dimgray"] if is_fr else ["r", "teal", "darkorange", "gold", "silver", "dimgray"], 
                fliersize=0)
    ax.set(ylim=y_lim, ylabel=crit_.split("_")[0].upper(), xlabel="Lead times (days)")
    ax.yaxis.set_major_locator(mtick.MultipleLocator(y_locator))
    ax.legend(ncols=n_col_lgd, loc="lower center" if "crps" not in crit_ else "upper center",
                facecolor="lightgrey", fontsize=12, columnspacing=0.2)
    ax.grid(ls="--", lw=0.3, c="grey")
    if "pers" in crit_.lower():
        ax.axhline(0., ls="--", c="r", lw=0.6)
    ax.tick_params(labelsize=12)
    plt.savefig("_ttt.png", bbox_inches="tight", dpi=300)
    plt.close(fig_x)
    return fig_x

In [ ]:
fg_mono,fg_bi =None, None
for bm_x in ["lstm", "sacsma"][:(1 if is_fr else None)]:
    for crit_x in ["pers", "nse", "kge"]:
        min_ = -.5 if "pers" in crit_x else 0.
        fg_mono = mono_boxplot_v0(res_df=res_pf, crit_=f"{crit_x}_on_median", bm_name=bm_x, y_lim=(min_,1.02), fig_s=(8,3))
        fg_bi = bi_boxplot_v0(res_df=res_pf, crit_=f"{crit_x}_on_median", bm_name=bm_x, y_lim=(min_,1.02))
        if SAVE_IMAGES:
            fg_mono.savefig(IMAGES_BOX+f"/{crit_x.upper()}_PERF_{bm_x.upper()}_mono_box_{CASE_CT}.png", bbox_inches="tight", dpi=300)
            fg_bi.savefig(IMAGES_BOX+f"/{crit_x.upper()}_PERF_{bm_x.upper()}_bi_box_{CASE_CT}.png", bbox_inches="tight", dpi=300)
fg_mono

In [ ]:
fg_bi

---
# C. Ensemble-based forecast post-processing
---

### C.1. Spread and Skill

In [ ]:
def plot_spread_skill_ratio(df_ssr, benchmark_case:str=None, save:bool=SAVE_IMAGES,
                            y:str = None, str_end:str = None, y_locator:float = 0.5):
    """
    Plot spread_skill ratio
    
    :param df_ssr: dataframe of the spread skill data
    :param benchmark_case: which of lstm or sacsma to consider
    :param save: whether to save directly the plot or not
    :param y: the name standing for the spread-skill column
    :param str_end: a string to identify the plot
    :param y_locator: set locators for y-axis
    
    :return: the figure object, can be saved also afterward
    """
    str_end = "" if str_end is None else f"_{str_end}"
    if y is None:
        y = "spread_skill"
    if benchmark_case is None:
        benchmark_case = "lstm"
    list_ctx_use = cx_list_dict[benchmark_case]
    fig, ax = plt.subplots(figsize=(8, 3))
    ax = sns.boxplot(df_ssr[df_ssr.index.get_level_values("context").isin(list_ctx_use)],
                x="hp", y=y, hue="context", ax=ax, 
                palette=[generic_style[c]["color2"] for c in list_ctx_use],
                hue_order=list_ctx_use, linecolor="k", linewidth=0.5,
                fliersize=0,)
    ax.set(ylim=(0., 2. if "sma" not in benchmark_case else 3.), xlabel="Lead times (days) ")
    ax.set_ylabel("$SSR = \\frac{Spread}{RMSE}$", fontsize=13)
    ax.yaxis.set_major_locator(mtick.MultipleLocator(y_locator))
    ax.set_xticklabels(labels=[f"hp={i}" for i in df_ssr.index.get_level_values("hp").unique()], fontsize=13)
    ax.tick_params(labelsize=11)
    
    ax.grid(ls="--", lw=".3", color="grey")
    ax.axhline(1, ls='--', color="k", lw=0.5)
    line_, label_ = custom_legend(axe=ax, dict_labels=ctx_name_dict0)
    ax.legend(line_, label_, title="", loc = "upper center",
               bbox_to_anchor=(.5, -.2), 
               borderaxespad=0, columnspacing=.3,
               ncol=len(list_ctx_use), 
               frameon=True, fontsize=11)
    if save is True:
        plt.savefig(f"{IMAGES_BOX}/Spread_skill_MLP_{benchmark_case.upper()}_by_hp{str_end}_{CASE_CT}.png", 
                    bbox_inches='tight', dpi=300)
    plt.close(fig)
    return fig

In [ ]:
def read_spread_skill(ens_type: str, nat_sc: pd.DataFrame = None) -> pd.DataFrame:
    """ read spread-skill scores and join with natural climatology if exist"""
    df = read_pqt(rf"{METRICS_MAIN_DIR}/postp_{ens_type}/pp_spread_skill_pandas.parquet.gzip")
    if nat_sc is not None:
        df = pd.concat([df, nat_sc], axis=0)
    return df

sp_sk = read_spread_skill("clim")
nat_rec = sp_sk.loc[sp_sk.index.get_level_values("context") == "natural_records"]
sp_sk_hind = read_spread_skill("hind", nat_rec)
sp_sk_real, sp_sk8 = None, None
if "us" in CASE_CT:
    sp_sk8 = read_spread_skill("clim8seeds")
if is_fr:
    sp_sk_real = read_spread_skill("real", nat_rec)
    
if "us" in CASE_CT:
    for bm_c in ["sacsma", "lstm"]:
        fg20 = plot_spread_skill_ratio(sp_sk, benchmark_case=bm_c, y="ssr", str_end="20seeds")
        fg8 = plot_spread_skill_ratio(sp_sk8, benchmark_case=bm_c, y="ssr", str_end="8seeds")
else:
    fg20 = plot_spread_skill_ratio(sp_sk, benchmark_case="lstm", y="ssr", str_end="20seeds")
fg20

In [ ]:
C_SSP = [sp_sk, sp_sk_hind, sp_sk_real, sp_sk] if "fr" in CASE_CT else [sp_sk, sp_sk_hind, sp_sk]
S_SSP = FORCAST_CASES[1:]+["P.O."]
benchmark_case = "lstm"
list_ctx_use_ = cx_list_dict[benchmark_case]

fig, axs = plt.subplots(1, len(C_SSP), figsize=(9, 2), sharey="row", width_ratios=[7,7,7,1] if is_fr else [7,7,1])
for i, df_ssr in enumerate(C_SSP):
    list_ctx_use = list_ctx_use_[:-1] if i<(len(C_SSP)-1) else list_ctx_use_[-1:]
    to_plot = df_ssr[df_ssr.index.get_level_values("context").isin(list_ctx_use)]
    to_plot = to_plot if i < (len(C_SSP)-1) else to_plot[to_plot.index.get_level_values("hp")==1]
    axi=axs[i]
    axi=sns.boxplot(to_plot,
                x="hp", y="ssr", hue="context", ax=axi, 
                palette=[generic_style[c]["color2"] for c in list_ctx_use],
                hue_order=list_ctx_use, linecolor="gray", linewidth=0.3,
                flierprops={"marker":".", "ms":3, "mew":0., "mfc":"k"}, 
                legend=True if i ==1 else False
                )
    axi.set_ylim(0., 2.)
    axi.set_title(f"{FCST_DICT.get(S_SSP[i], 'P.O.').title()}")
    axi.axhline(1, ls='--', color="k", lw=0.4)
    axi.set_ylabel("SSR" if i==0 else None, fontsize=11)
    if i==(len(C_SSP)-1):
        axi.set_xticklabels(labels=["1 to 7"], fontsize=11)
        axi.set_xlabel(None)
    else:
        axi.set_xlabel("Lead times (days) ", fontsize=11)
    axi.grid(ls="--", lw=.3, color="grey")
    axi.tick_params(labelsize=11)
    axi.yaxis.set_major_locator(mtick.MultipleLocator(0.5))
    if i==1:
        axi.legend(handles=[mpatches.Patch(color=generic_style[c]["color2"], 
                                           label=ctx_name_dict0[c]) 
                            for c in list_ctx_use_], 
                   loc = "upper center", 
                   columnspacing=0.7,
                   bbox_to_anchor=(.5 if is_fr else 0., -.3), ncol=5)
plt.subplots_adjust(wspace=0.15)
if SAVE_IMAGES is True:
    fig.savefig(f"{IMAGES_BOX}/Spread_skill_3sub_MLP_{benchmark_case.upper()}_{CASE_CT}.png", 
                bbox_inches='tight', dpi=300)

### C.2. CRPS

In [ ]:
def read_regr_pp(ens_type: str, nat_rec: pd.DataFrame = None) -> pd.DataFrame:
    """ Read regression type metrics including CRPS"""
    path = rf"{METRICS_MAIN_DIR}/postp_{ens_type}"
    df = pd.concat([read_pqt(os.path.join(path, f))
                    for f in ["pp_metrics_pandas.parquet.gzip", "pp_crps_pandas.parquet.gzip"]],
                   axis=1)
    if nat_rec is not None:
        df = pd.concat([df, nat_rec], axis=0)
    return df

res_clim = read_regr_pp("clim")
nat_rec  = res_clim.loc[res_clim.index.get_level_values("context") == "natural_records"]
res_hind = read_regr_pp("hind", nat_rec)
res_real = None
if is_fr:
    res_real = read_regr_pp("real", nat_rec)

In [ ]:
# The crps for a single is assimilated as its MAE score
MAE_NAIVE = obs_naive.groupby(["basin", "hp"]).apply(lambda x: mean_absolute_error(x.obs, x.naive)).to_frame("crps")
MAE_NAIVE["context"] = "persistent"
MAE_NAIVE = MAE_NAIVE.set_index("context", append=True).reorder_levels(["context", "basin", "hp"]).sort_index()
MAE_NAIVE = MAE_NAIVE.loc[MAE_NAIVE.index.get_level_values("basin").isin(res_clim.index.get_level_values("basin").unique())]

In [ ]:
# Plot
C_CRPS = [res_clim, res_hind, res_clim] if not is_fr else [res_clim, res_hind,res_real, res_clim]
S_ALIAS = FORCAST_CASES[1:]+["P.O."]
benchmark_case="lstm"

fig, axs = plt.subplots(1, len(C_CRPS), figsize=(9, 2), sharey="row", width_ratios=[7,7,7,1] if is_fr else [7,7,1])
for i, (c_res, fcst_c) in enumerate(zip(C_CRPS, S_ALIAS)):
    c_res_naive = pd.concat([c_res[["crps"]].sort_index(), MAE_NAIVE.sort_index()], axis=0)
    list_ctx_use_ = cx_list_dict[benchmark_case]
    list_ctx_use = list_ctx_use_[:-1] + ["persistent"]  if i<(len(C_CRPS)-1) else list_ctx_use_[-1:]
    to_plot = c_res_naive[c_res_naive.index.get_level_values("context").isin(list_ctx_use)]
    to_plot = to_plot if i < (len(C_CRPS)-1) else to_plot[to_plot.index.get_level_values("hp")==3]
    axi=axs[i]
    axi=sns.boxplot(to_plot,
                    x="hp", y="crps", hue="context", ax=axi, 
                    palette=[generic_style[c]["color2"] for c in list_ctx_use],
                    hue_order=list_ctx_use, linecolor="gray", linewidth=0.3,
                    fliersize=0, 
                    legend=True if i ==1 else False)
    
    axi.set(ylim=(0., 2. if is_fr else 3.),
            ylabel="CRPS" if i==0 else None, 
            title=f"{FCST_DICT.get(S_ALIAS[i], 'P.O.').title()}")
    
    if i==(len(C_CRPS)-1):
        axi.set_xticklabels(labels=["1 to 7"], fontsize=11)
        axi.set_xlabel(None)
    else:
        axi.set_xlabel("Lead times (days) ", fontsize=11)
    axi.grid(ls="--", lw=.3, color="grey")
    axi.tick_params(labelsize=11)
    axi.yaxis.set_major_locator(mtick.MultipleLocator(1.))
    if i==1:
        axi.legend(handles=[mpatches.Patch(color=generic_style[c]["color2"], 
                                           label=ctx_name_dict0[c]) 
                            for c in list_ctx_use_[:-1]+["persistent"] + list_ctx_use_[-1:]], 
                   loc = "lower center",
                   columnspacing=0.6, 
                   bbox_to_anchor=(.1 if not is_fr else .5 , -.5), 
                   ncol=6)
plt.subplots_adjust(wspace=0.1)
if SAVE_IMAGES is True:
    fig.savefig(f"{IMAGES_BOX}/CRPS_3sub_MLP_{benchmark_case.upper()}_clim_hind_real_{CASE_CT}.png", 
                bbox_inches='tight', dpi=300)

#### Gain of CRPS

In [ ]:
if is_fr:
    ALL_CSRPS = pd.concat([a["crps"].to_frame(FCST_DICT.get(["clim", "hind", "real"][i])) for i, a in enumerate([res_clim, res_hind, res_real])], axis=1)
    fgg, axs=plt.subplots(1, 2, figsize=(8,2.5), sharey="row")
    for i, c in enumerate(["clim", "hind", "real"][1:]):
        ax=axs[i]
        to_pt = (ALL_CSRPS[FCST_DICT.get(c)] - ALL_CSRPS[FCST_DICT.get("clim")]).to_frame(FCST_DICT.get(c))
        to_pt = to_pt.loc[~to_pt.index.get_level_values("context").str.startswith("natural")].rename(index =ctx_name_dict0, level="context")
        sns.boxplot(to_pt, x="hp", y= FCST_DICT.get(c), hue="context",  
                    ax=ax, linewidth=0.3, width=0.6,
                    flierprops=fliers_props, 
                    palette=["r", "teal"])
        ax.set(ylim=(-.6,.6), 
               ylabel="Excess of CRPS \n against the Climatology \n (X - Clim.)", 
               xlabel="Lead times (days)", 
               title=f"{FCST_DICT.get(c)}")
        ax.axhline(0., ls="--", color="k", lw=0.6)
        ax.legend(loc="lower left", ncols=2)
        ax.grid(axis="y", ls="--", lw=0.3, color="k")
    plt.subplots_adjust(wspace=0.1)
    if SAVE_IMAGES is True:
        plt.savefig(IMAGES_BOX+f"/CRPS_Diff_among_Forecast_Products_bx_{CASE_CT}.png", dpi=300, bbox_inches="tight")

#### CRPSS

In [ ]:
x_cas = "clim"
if "fr" in CASE_CT:    
    ALL_CSRPS = pd.concat([a["crps"].to_frame(FCST_DICT.get(["clim", "hind", "real"][i])) for i, a in enumerate([res_clim, res_hind, res_real])], 
                          axis=1)
    fgg, axs=plt.subplots(1, 2, figsize=(8,2.5), sharey="row")
    for i, c in enumerate([c for c in ["clim", "hind", "real"] if c!=x_cas]):
        ax=axs[i]
        to_pt = (1 - ALL_CSRPS[FCST_DICT.get(c)] / ALL_CSRPS[FCST_DICT.get(x_cas)]).to_frame(FCST_DICT.get(c))
        to_pt = to_pt.loc[~to_pt.index.get_level_values("context").str.startswith("natural")].rename(index =ctx_name_dict0, level="context")
        sns.boxplot(to_pt, x="hp", y= FCST_DICT.get(c), hue="context",  ax=ax, linewidth=0.3, width=0.6, flierprops=fliers_props, palette=["r", "teal"])
        ax.set(ylim=(-1.,1.), 
               ylabel=f"CRPSS = "+ r"$1 - \frac{CRPS_X.}{CRPS_{REF.}}$", 
               xlabel="Lead times (days)", 
               title=f"1 - {FCST_DICT.get(c).title()}/{x_cas.title()}.")
        ax.axhline(0., ls="--", color="k", lw=0.6)
        ax.legend(loc="lower left", ncols=2)
        ax.grid(axis="y", ls="--", lw=0.3, color="k")
        ax.yaxis.set_major_locator(mtick.MultipleLocator(.5))
    plt.subplots_adjust(wspace=0.1)
    if SAVE_IMAGES is True:
        plt.savefig(IMAGES_BOX+f"/CRPSS_{x_cas.upper()}_Vs_Forecast_Products_bx_{CASE_CT}.png",
                    dpi=300, bbox_inches="tight")

### C.3. Rank Diagram

#### C.3.1 For past observe discharge

In [ ]:
## Rank diagram Q
natural_Q = pd.read_parquet(BASE_PP + "/natural_records_hp1_CLIM56.parquet.gzip", engine="pyarrow")
natural_Q = natural_Q.join(obs_naive[["obs"]], on=["hp", "basin", "Date"]).dropna(axis=0).sort_index(axis=1)
natural_Q= natural_Q.swaplevel("Date", "year").sort_index()

In [ ]:
RANK_CLASSES = len(natural_Q.index.get_level_values("year").unique()) + 1
basin_rank_hist = {}
for basin, df_b in natural_Q.groupby("basin"):
    ranks_q = []
    for date, group in df_b.groupby("Date"):
        preds = group.sort_values("year")["pred"].values
        obs = group["obs"].iloc[0]
        rank = np.sum(preds < obs)  # Ranks of obs
        ties = np.sum(preds == obs) # Tie
        if ties > 0:
            rank += np.random.randint(0, ties + 1)
        ranks_q.append(rank)
    hist, _ = np.histogram(ranks_q, bins=np.arange(RANK_CLASSES + 1) - 0.5)
    hist = hist / hist.sum()
    basin_rank_hist[basin] = hist
rank_nat_q = pd.DataFrame.from_dict(basin_rank_hist, orient="index", columns=[i+1 for i in range(RANK_CLASSES)])

fig_nat, ax = plt.subplots(figsize=(8, 4))
sns.barplot(rank_nat_q, ax=ax, capsize=0.3, width=0.8, err_kws={"lw":.5, "color":"k"}, palette=["teal"])
ax.axhline(1/RANK_CLASSES, ls="--", color="k", lw=0.5)
ax.set(ylabel="Frequency",xlabel="Ranks")
ax.grid(axis="y", linestyle="--", alpha=0.5)
ax.tick_params(labelsize=10)
if SAVE_IMAGES is True:
    fig_nat.savefig(IMAGES_BOX+f"/Rank_Diagram_Natural_Q_{CASE_CT}.png", dpi=300, bbox_inches="tight")

#### C.3.2. For forecasted discharges

In [ ]:
RK_FCST = FORCAST_CASES[1:]
for rk_c_ctx in ctx_pair_list:
    benchmark_c = rk_c_ctx[0]
    fig_rk, axs = plt.subplots(len(rk_c_ctx), len(RK_FCST), 
                               figsize=(9, 3) if is_fr else (6, 5), 
                               sharex="col", edgecolor=".5", 
                               facecolor="w", sharey="row",)
    for d, fcst_c in enumerate(RK_FCST):
        df_rkk = read_pqt(os.path.join(rf"{METRICS_MAIN_DIR}/postp_{fcst_c}", "pp_ranks_pandas.parquet.gzip"))
        n_bv = len(df_rkk.index.get_level_values("basin").unique())
        for i, cx_rk in enumerate(rk_c_ctx):
            ax=axs[i,d]
            rank_hp_bv = df_rkk.loc[(df_rkk.index.get_level_values("context") == cx_rk) , :].droplevel(level=["context"])
            sns.barplot(data=rank_hp_bv.reset_index(), x="rank_bins", y="histogram_rank", 
                        hue="hp", palette="rocket_r",
                        lw=0.3, edgecolor="gray",
                        capsize=0.2, 
                        err_kws={"color": "k", "lw": .4},
                        ax=ax, width=0.80, 
                        legend=True, )
            if i ==0:
                ax.set_title(f"{FCST_DICT.get(fcst_c).title()}", y=0.95, fontsize=12)
            if i+d!=(len(RK_FCST)+len(rk_c_ctx) - 2):
                ax.legend().remove()
            ax.set_xlim(xmax=9.5)
            ax.set_xlabel("Rank bins", fontsize=10)
            ax.set_ylabel(f"Ratio ({n_bv})" if d==0 else None, fontsize=10)
            ax.set_ylim(ymax=0.4)
            
            ax.axhline(0.1, ls="--", lw=0.5, c="k")
            ax.text(x=3.5,y=.25, s=generic_style[cx_rk]['short_0'], size=9)
            ax.tick_params(axis='both', which='major', labelsize=10)
            ax.yaxis.set_major_locator(MultipleLocator(0.1))
                
        sns.set_style("ticks")
        plt.rcParams['xtick.major.size'] = 2
        plt.legend(title="Forecasting lead time (days)", 
                   bbox_to_anchor=(0.2, -.45) if is_fr else (.85, -.4), 
                   columnspacing=0.3,
                   borderaxespad=0, fontsize=10,
                   ncol=7, frameon=False)
    
    plt.subplots_adjust(hspace=0.15, wspace=0.08)
    if SAVE_IMAGES:
        plt.savefig(f"{IMAGES_BOX}/rank_3sub_all_hp_by_ctx_{benchmark_c.upper()}_{CASE_CT}.png", bbox_inches='tight', dpi=300)

### C.4. Brier SCORE

In [ ]:
def read_brier_pp(ens_type: str, nat_clim_sc: pd.DataFrame = None) -> pd.DataFrame:
    """Read brier scores from metric dir, with ens_type in [hind, clim, real] for one of hindcast,
     climatology or forecast archives respectively"""
    path = rf"{METRICS_MAIN_DIR}/postp_{ens_type}"
    up  = read_pqt(os.path.join(path, "pp_briers_pandas.parquet.gzip"))
    blw = read_pqt(os.path.join(path, "pp_briers_ie_pandas.parquet.gzip"))
    blw.columns = ["brier"]
    df = pd.concat([
                    up.loc[up.index.get_level_values("quantile").isin(FLOOD_QT)],
                    blw.loc[blw.index.get_level_values("quantile").isin(DROUGHT_QT)],
                ], axis=0)
    if nat_clim_sc is not None:
        df = pd.concat([df, nat_clim_sc], axis=0)
    return df


brier_clim = read_brier_pp("clim")
nat_ens = brier_clim.loc[brier_clim.index.get_level_values("context") == "natural_records"]
brier_hind = read_brier_pp("hind", nat_ens)
brier_real = None
if "fr" in CASE_CT:
    brier_real = read_brier_pp("real", nat_ens)

#### Plot

In [ ]:
msz = [7, 6, 6, 7, 9, 7, 7]
n_bv = len(list_basin_56)
C_BRIER = [brier_clim, brier_hind, brier_real][:(None if is_fr else -1)]
fcst_labels =  FORCAST_CASES[1:]

for ctx_case in ctx_pair_list:
    benchmark_c=ctx_case[0]
    ctx_case += ("natural_records",)
    fig_bs, axs = plt.subplots(len(list_hp3), len(fcst_labels), 
                               figsize=(9.5, 6) if is_fr else (7, 5), 
                               sharey="all", sharex=True,
                               edgecolor=".5", facecolor="w")
    for d, (brier, fcst_c) in enumerate(zip(C_BRIER,fcst_labels)):
        for h_, hp_bs in enumerate(list_hp3):
            ax=axs[h_, d]
            for i, ctk in enumerate(ctx_case):
                brier_ctx_hp_ = (brier
                                 .loc[(brier.index.get_level_values("hp") == hp_bs)&(brier.index.get_level_values("context") == ctk) , :]
                                 .droplevel(level=["hp", "context", "basin"]))
                brier_ctx_hp_.columns=[ctx_name_dict0[ctk]]
                _= sns.lineplot(brier_ctx_hp_,ax=ax, 
                                markers = generic_style[ctk]["marker_rel"], ms=msz[i], 
                                mec="k", mfc=generic_style[ctk]["color_id"],
                                mew=0.3, lw=0.2, palette=["k"], err_style="bars", err_kws={"lw":.8},)
            if h_*d!=(len(C_BRIER)+len(list_hp3)-2):
                ax.get_legend().remove()
            ax.set(xlim=(0.,1.), ylim=(0., .3), xlabel=r"Discharge threshold ($Q_x$)", 
                   ylabel=f"Hp [days] = {hp_bs} \n Brier's ({n_bv} basins)",
                   title=f"{FCST_DICT.get(fcst_c).title()}" if h_ == 0 else "")
            ax.set_xticks(ticks=USED_QT, labels=USED_QT)
            ax.tick_params(axis='both', which='major', labelsize=10)
            
            ax.axvline(0.5, ls="--", color="k", lw=0.5)
            ax.axhline(0.25, ls="--", color="r", lw=0.5)
            
            ax.text(0.11, 0.255, r"Random detection model", color="gray", style="italic")
            ax.text(0.5- 0.27, 0.02, r"q$\leq{Q_x}$", color="k")
            ax.text(0.5+0.1, 0.02, r"q$>Q_x$", color="k")
            
            ax.yaxis.set_major_locator(set_step_locator(0.1))
            ax.grid(lw=0.1)
        leg_elt=[]
        for e, ln in enumerate(ax.lines[::3]):
            leg_elt.append(Line2D([0], [0], color=ln.get_c(), ls=ln.get_ls(), ms=ln.get_ms(), lw=ln.get_lw(), 
                                  markerfacecolor=ln.get_mfc(),
                                  marker=ln.get_marker()))
        plt.legend(leg_elt, 
                   [generic_style[c]["short_0"] for c in ctx_case], 
                   loc="lower center", 
                   bbox_to_anchor=(-.6, -.6) if is_fr else (-.13, -.7),
                   columnspacing=0.2,
                   ncol=5, )
        plt.subplots_adjust(wspace=0.1, hspace=0.1)
        plt.rcParams['xtick.major.size'] = 2
        if SAVE_IMAGES:
            plt.savefig(f"{IMAGES_BOX}/Brier_2boxes_CLIM_n_HIND_{benchmark_c}_{CASE_CT}.png", bbox_inches='tight', dpi=300)

### C.5.1 ROC

In [ ]:
def reinterp_roc(roc: pd.DataFrame, n_points: int = 500) -> pd.DataFrame:
    """Interpolate ROC"""
    roc_ = roc.drop_duplicates("FP")
    interp = scipy.interpolate.interp1d(x=roc_["FP"].values,
                                        y=roc_["TP"].values,
                                        axis=0)
    x = np.linspace(0, 1, n_points, endpoint=True)
    return pd.DataFrame(data={"FP": x,
                              "TP": interp(x)}).set_index("FP")

def read_roc_pp(ens_type:str, nat_sc:pd.DataFrame = None):
    """Read roc scores on ensemble types from [clim, hind, real]"""
    x_pp_path =rf"{METRICS_MAIN_DIR}/postp_{ens_type}"
    _up = read_pqt(os.path.join(x_pp_path, f"pp_rocs_pandas.parquet.gzip"))
    _blw = read_pqt(os.path.join(x_pp_path, f"pp_roc_ie_pandas.parquet.gzip"))
    
    _up = _up.loc[_up.index.get_level_values("quantile").isin(FLOOD_QT)]
    _blw = _blw.loc[_blw.index.get_level_values("quantile").isin(DROUGHT_QT)]
    all_ = pd.concat([_up, _blw], axis=0)
    if nat_sc is not None:
        all_ = pd.concat([all_, nat_sc], axis=0)
    return all_

roc_clim = read_roc_pp("clim")
roc_natural = roc_clim[roc_clim.index.get_level_values("context")=="natural_records"]
roc_hind = read_roc_pp("hind", roc_natural)
roc_real = None
if is_fr:
    roc_real = read_roc_pp("real", roc_natural)

### C.5.2. AUC 

In [ ]:
AUC_QT = USED_QT
def compute_auc_from_roc(rocs: pd.DataFrame, qt_list: list) -> pd.DataFrame:
    """Compute the AUC scores from the ROC dataframe"""
    auc = rocs.groupby(["context", "hp", "basin", "quantile"]).agg({"AUC": ["median"]})
    auc.columns = ["auc_med"]
    return auc.loc[auc.index.get_level_values("quantile").isin(qt_list)]

auc_clim, auc_hind = [compute_auc_from_roc(r, AUC_QT) for r in [roc_clim, roc_hind]]
auc_real = None
if is_fr:
    auc_real = compute_auc_from_roc(roc_real, AUC_QT)

In [ ]:
C_AUC = [auc_clim, auc_hind, auc_real][:(None if is_fr else -1)]
fcst_labels = S_AUC= FORCAST_CASES[1:]

for ctx_case in ctx_pair_list:
    benchmark_c=ctx_case[0]
    ctx_case += ("natural_records",)
    fig_auc, axs = plt.subplots(len(list_hp3), len(fcst_labels), 
                                figsize=(8, 5) if is_fr else (7, 5), 
                                sharey=True, sharex=True)
    for d, (AU_X, fcst_c) in enumerate(zip(C_AUC, fcst_labels)):
        n_bv = len(AU_X.index.get_level_values("basin").unique())
        for h_, hp in enumerate(list_hp3):
            AU_X_ = AU_X.loc[AU_X.index.get_level_values("hp")==hp].droplevel(level=["hp"])
            ax = axs[h_,d]
            for c_, ccx in enumerate(ctx_case):
                auc_hp_cx = AU_X_.loc[AU_X_.index.get_level_values("context")==ccx].droplevel(level=["context", "basin"])
                auc_hp_cx.columns=[ctx_name_dict0[ccx]]   
                sns.lineplot(auc_hp_cx, 
                             markers=generic_style[ccx]["marker_rel"], 
                             ms=msz[c_], mew=0.8, mec="k", 
                             mfc=generic_style[ccx]["color_id"],
                             lw=.3, ax=ax, palette=["k"],
                             err_style="bars", err_kws={"lw":.8}, )
            if h_*d!=(len(S_AUC)+len(list_hp3)-2):
                ax.get_legend().remove()
                
            ax.set(ylim=(0.45, 1.), xlim=(0., 1.),
               xlabel=r"Discharge threshold ($Q_x$)",ylabel=f"Hp [days] = {hp}\nAUC ({n_bv} basins)",
               title=FCST_DICT.get(fcst_c).title() if h_ == 0 else "")                
            ax.set_xticks(ticks=AUC_QT, labels=AUC_QT)
            ax.tick_params("both", labelsize=10)
            
            ax.text(0.23, 0.6, r"q$\leq{Q_x}$", color="k")
            ax.text(0.6, 0.6, r"q$>Q_x$", color="k")
            ax.text(0.12, 0.46, r"Random detection model", color="gray", style="italic")
            ax.axvline(0.5, ls="--", color="k", lw=0.5)
            ax.axhline(0.5, ls="--", color="r", lw=0.5)
            ax.grid(lw=0.2)
            ax.yaxis.set_major_locator(set_step_locator(0.1))
            
    leg_elt=[]
    leg_ax = [l for l in ax.lines if not l.get_label().startswith("_")]
    for e, ln in enumerate(leg_ax):
        leg_elt.append(Line2D([0], [0], color=ln.get_c(), ls=ln.get_ls(), ms=ln.get_ms(), lw=ln.get_lw(), 
                              markerfacecolor=ln.get_mfc(),
                              marker=ln.get_marker()))
    plt.legend(leg_elt, 
               [generic_style[c]["short_0"] for c in ctx_case],
               loc="lower center", 
               bbox_to_anchor=(-.1 if not is_fr else -.6, -.7),
               columnspacing=.2, ncol=5)
    plt.subplots_adjust(wspace=0.1, hspace=0.1)
    if SAVE_IMAGES:
        plt.savefig(f"{IMAGES_BOX}/AUC_2box_hp_CLIM_n_HIND_{benchmark_c}_{CASE_CT}.png", bbox_inches='tight', dpi=300)

---
## D. SPECIAL METRICS
---

In [ ]:
if "us" in CASE_CT:
    res_perf_all = pd.read_parquet(METRICS_MAIN_DIR + f"/postp_perf/pp_metrics_pandas.parquet.gzip", engine="pyarrow")
    BM_PERS = res_perf_all.loc[res_perf_all.index.get_level_values("context").isin(["lstm", "sacsma"])]
    res_perf = res_perf_all.loc[~res_perf_all.index.get_level_values("context").isin(["lstm", "sacsma"])]
    pers_cx = res_perf["pers_on_median"]
    pers_bm = BM_PERS["pers_on_median"]
    diff_pair_list = [("no_in_mlp", "sacsma"), ("sma_in_mlp", "sacsma"), ("predict_sma_error", "sacsma"),
                      ("no_in_mlp", "lstm"), ("lstm_in_mlp", "lstm"), ("predict_lstm_error", "lstm")]
   
    class_PERS_BM = BM_PERS.copy()
    c=[-10000, 0., 0.5, 1.]

    class_PERS_BM["PERS_class"] = np.nan
    cl_=[rf"(-$\infty$, {c[1]}]", f"({c[1]}, {c[2]}]",  f"({c[2]}, {c[3]}]"]
    for i in range(len(c)-1):
        b=c[i+1]
        class_PERS_BM.loc[(c[i] < class_PERS_BM["pers_on_median"]) & (class_PERS_BM["pers_on_median"] <= b),  "PERS_class"]=cl_[i]
    class_PERS_BM = class_PERS_BM["PERS_class"].to_frame()
    
    # Plotting
    _, axx = plt.subplots(3, 2, figsize=(8, 4), sharey="row", sharex="col")
    for ibm, dif_bm in enumerate([diff_pair_list[:3], diff_pair_list[3:]]):
        dif_cr_lst = []
        for i, hpx in enumerate([1, 3, 7]):
            ax = axx[i, ibm]
            dif_cr_df = pd.DataFrame()
            for c_, (ccx_, bm_) in enumerate(dif_bm):
                dif_cr_df[generic_style[ccx_]["short"]]=pers_cx.xs(ccx_, level="context").xs(hpx, level="hp") - pers_bm.xs(bm_, level="context").xs(hpx, level="hp")
            class_temp = class_PERS_BM.loc[(class_PERS_BM.index.get_level_values("context") == bm_) & (class_PERS_BM.index.get_level_values("hp") == hpx)]
            dif_cr_df["class_PERS"] = class_temp.droplevel(["context", "hp"]).loc[dif_cr_df.index, "PERS_class"]
            dif_df2 = dif_cr_df.melt(ignore_index=False, value_name="gain", var_name="context", id_vars=["class_PERS"])
            dif_df2.sort_values(by=["context", "class_PERS"], ascending=[False, True], inplace=True)
            dif_df2 = dif_df2.loc[dif_df2["gain"] <= 1.5]
            sns.boxplot(dif_df2, x="class_PERS", y="gain", hue="context", linecolor="k", fliersize=0., ax=ax, linewidth=0.6,
                        palette="Oranges_r")
            if i == 2:
                ax.legend(title="DA approach", fontsize=10, loc="lower right", ncols=3)
                ax.set_xlabel(f"PERS-class on {['SAC-SMA', 'LSTM'][ibm]}")
            else:
                ax.get_legend().remove()
                ax.set_xlabel("")
            ax.set_ylim(-1., 2.)
            ax.axhline(0., ls="--", lw=0.5, color="k")
            ax.grid()
            ax.set_ylabel(f"Hp:{hpx} \nGain $\pm$" if ibm == 0 else "")
            if i == 0:
                ax.set_title(f"PERS gain on {['SAC-SMA', 'LSTM'][ibm]}", fontsize=10)
        sns.move_legend(ax, "upper center", bbox_to_anchor=(.5, -.5), ncol=3)
    plt.subplots_adjust(wspace=0.1)
    if SAVE_IMAGES:
        plt.savefig(f"{IMAGES_BOX}/Gain_Pers_on_PERS_Class_b2_{CASE_CT}.png", bbox_inches="tight", dpi=300)

---
# E. Zoom on forecasted hydrographs
---

In [ ]:
RAW_PARQUET = BASE_PP
HYDRO_TO = f"{IMAGES_BOX}/hydro_zoom"
os.makedirs(HYDRO_TO, exist_ok=True)
ZOOM_BV_X = {"us": {"zoom":(523,584), "basin":"01055000"}, "fr": {"zoom": (520, 570), "basin": "K132181010"}}
zoom_=ZOOM_BV_X.get(CASE_CT)["zoom"]
basin = bv_x =ZOOM_BV_X.get(CASE_CT)["basin"]
levels_order = ["context", "hp", "basin", "year", "seed", "Date"]

### Prepare plotting

In [ ]:
# Shared palette
_RIBBON_PALETTE = ["silver"] + sns.color_palette("crest", n_colors=5)
_RIBBON_QTS = [(0., "Min-Max"), (0.025, "Q95"), (0.125, "Q75")]

def _fmt_date_ticks(ax, step_x: int = 5):
    """Reformat x labels into 'mm/dd', spaced by step_x."""
    labs = [t.get_text() for t in ax.get_xticklabels()]
    try:
        short = [datetime.strptime(l, "%Y-%m-%d").strftime("%m\n%d") for l in labs]
        ticks = list(range(0, len(short), step_x))
        ax.set_xticks(ticks)
        ax.set_xticklabels([short[i] for i in ticks])
    except ValueError:
        _=ax.set_xticklabels(labs)
        return

def _style_pred_ax(ax):
    """Common style to show predictions."""
    ax.legend(loc="upper left", ncols=2, fontsize=8)
    ax.grid(ls="--", lw=0.1, color="grey")


def plot_pred_w_box(plotab, ax, y="d_rl", step_x=5, obs_c="k", ms=5):
    """Plot discontinuous prediction using box-plot"""
    to_pt = plotab.dropna()
    sns.boxplot(to_pt, x="Date", y=y, width=.6, color="teal", ax=ax,
                linewidth=0.5, linecolor="k", label="Pred",
                flierprops={"marker": ".", "ms": 4, "mew": 0., "mfc": "gray"})
    ax.plot(to_pt["obs"].xs(1, level="year").xs(1, level="seed").values,
            marker=".", lw=0., ms=ms, color="k", mfc=obs_c, mew=0.1, label="Obs")
    _fmt_date_ticks(ax, step_x)
    _style_pred_ax(ax)
    return ax


def plot_continue_pred(plotab, ax, y="d_rl", step_x=5, obs_c="k", ms=5):
    """Plot continuous prediction normally """
    pr_p = (plotab[[y]].unstack(["year", "seed"])
                       .rename_axis("Date")
                       .pipe(lambda df: df.set_index(pd.to_datetime(df.index, format="%Y-%m-%d")))
                       .interpolate(axis=0))
    for i, (qt, name) in enumerate(_RIBBON_QTS):
        ax.fill_between(pr_p.index,
                        pr_p.quantile(qt, axis=1),
                        pr_p.quantile(1. - qt, axis=1),
                        label=name, color=_RIBBON_PALETTE[i * 2], lw=0.)
    obs_lev = plotab.index.get_level_values("year").min()
    ax.plot(plotab["obs"].xs(obs_lev, level="year").xs(1, level="seed"),
            marker=".", lw=0.1, ms=ms, color="k", mfc=obs_c, mec="k", mew=0.1, label="Obs")
    _style_pred_ax(ax)
    return ax


#### Gather preds

In [ ]:
def get_pred_ens_on_bv(list_hp_file, basin_list=None, lev_ord = levels_order):
    if basin_list is None:
        basin_list = [basin]
    all_p = []
    FILE_BAR = tqdm(list_hp_file, leave=True, desc="By-File")
    for f_c in FILE_BAR:
        FILE_BAR.set_postfix_str(Path(f_c).name)
        fc = pd.read_parquet(f_c, engine="pyarrow")
        fc = fc.loc[fc.index.get_level_values("basin").isin(basin_list)]
        if "seed" in fc.columns:
            fc = fc.set_index("seed", append=True)
        fc = fc.reorder_levels(lev_ord)
        fc.index = fc.index.set_levels(fc.index.levels[-2].astype(np.int8), level=-2)
        all_p.append(fc)
    all_p = pd.concat(all_p, axis=0).sort_index().dropna(axis=0)
    return all_p

def get_all_bv_all_hp_case(file_list:list, list_hp_=None, bv_list=None):
    """Filter predictions on a list of lead times and basins"""
    if list_hp_ is None:
        list_hp_ = [1, 3, 7]
    if bv_list is None:
        bv_list = [basin]
    hp_df_bvs = []
    for hp_i in list_hp_:
        hp_f = [f for f in file_list if f"_hp{hp_i}_" in f]
        hp_dfs = get_pred_ens_on_bv(hp_f, basin_list=bv_list)
        hp_df_bvs.append(hp_dfs)
    hp_df_bvs = pd.concat(hp_df_bvs, axis=0).sort_index()
    return hp_df_bvs

### Gather and group predictions in respect of the forecast-types

In [ ]:
all_lstm_cases = glob(RAW_PARQUET + f"/*lstm_*.parquet.gzip")
all_mlp_cases = glob(RAW_PARQUET + f"/no_in_mlp*.parquet.gzip")
all_lstm_mlp_cases = all_lstm_cases + all_mlp_cases

if "us" in CASE_CT:
    all_sma_cases = glob(RAW_PARQUET + f"/*sma_*.parquet.gzip")
    all_lstm_mlp_cases = all_lstm_mlp_cases + all_sma_cases
    
clim_cx = [f for f in all_lstm_mlp_cases if "clim56" in f.lower()]
perf_cx = [f for f in all_lstm_mlp_cases if f"perf{531 if 'us' in CASE_CT else 338}" in f.lower()]
hind_x = [f for f in all_lstm_mlp_cases if "hind56" in f.lower()]

real_x = None
if "fr" in CASE_CT:
    real_x = [f for f in all_lstm_mlp_cases if "real56" in f.lower()]

# GET PREDICTIONS
all_pf = get_all_bv_all_hp_case(perf_cx)
all_cl = get_all_bv_all_hp_case(clim_cx)
all_hd = get_all_bv_all_hp_case(hind_x)
all_real = None
if "fr" in CASE_CT:
    all_real = get_all_bv_all_hp_case(real_x)
    
all_cl = all_cl[all_cl.index.get_level_values("year")>0]

In [ ]:
def get_full_plotable(bv:str, hp_x:int=3):
    """Get pred for a given basin and a lead time"""
    df_ob = obs_naive["obs"].xs(bv, level="basin").xs(hp_x, level="hp")
    if "fr" in CASE_CT:
        thread_ = pd.concat([d_u["pred"].xs(bv, level="basin").xs(hp_x, level="hp").to_frame(f"d_"+["cl", "hd", "rl"][i]) \
                             for i, d_u in enumerate([all_cl, all_hd,  all_real])], axis=1, join="outer").sort_index()
    else:
        thread_ = pd.concat([d_u["pred"].xs(bv, level="basin").xs(hp_x, level="hp").to_frame(f"d_"+["cl", "hd"][i]) \
                             for i, d_u in enumerate([all_cl, all_hd])], axis=1, join="outer").sort_index()
    d_pf = all_pf["pred"].xs(bv, level="basin").xs(hp_x, level="hp").to_frame("d_pf").sort_index()
    d_pf.index = d_pf.index.set_levels(d_pf.index.get_level_values("year").unique().map({-1:1}), level="year")
    four_ = pd.concat([thread_, d_pf[~d_pf.index.duplicated(keep="first")]], axis=1)
    full_plot = four_.join(df_ob, on="Date")
    return full_plot, df_ob

#### Plot all predictions

In [ ]:
is_fr = "fr" in CASE_CT
N_ROWS, N_COLS = (4, 2 if is_fr else 3)
y_max, y_step = (15., 3.) if is_fr else (72., 12.)
str_rep = f"\nBasin N° {basin}"
fcst_keys = ["d_pf", "d_cl", "d_hd", "d_rl"] if is_fr else ["d_pf", "d_cl", "d_hd"]
FCST_Name = {"d_pf": "Perfect", "d_cl": "Climatology","d_hd": "Hindcast", "d_rl":  "Forc. Archives"}

OBS_C = "darkorange"
MkS = 7

for HP_X in list_hp3:
    for BENCHMARK in ["lstm", "sacsma"][: (1 if is_fr else None)]:
        MODEL_LABELS = [f"{BENCHMARK.upper()}", "MLP", f"$i$-{BENCHMARK.upper()}", f"$\\epsilon$-{BENCHMARK.upper()}"]
        full_plottable, df_obs = get_full_plotable(basin, HP_X)
        
        zoom_bv=(zoom_[0]-HP_X-1, zoom_[1]+(7-HP_X+1))
        index_r = df_obs.index[zoom_bv[0]:zoom_bv[1]]
        mod_keys = ctx_pair_list[-1 if "lstm" in BENCHMARK else 0] 
                
        fg, axs = plt.subplots(N_ROWS, N_COLS, figsize=(8 if is_fr else 10, 10), sharey="all")
        period_  = index_r
        
        for ri, (row_key, col_key) in enumerate(
                [(r, c) for r in (fcst_keys if is_fr else mod_keys)
                        for c in (mod_keys  if is_fr else fcst_keys)]):
            r, c   = divmod(ri, N_COLS)
            ax     = axs[r, c]
            d_pt   = row_key if is_fr else col_key
            m_d    = col_key if is_fr else row_key
        
            to_pt  = (full_plottable[[d_pt, "obs"]]
                      .xs(m_d, level="context")
                      .reorder_levels([-1, 0, 1])
                      .sort_index()
                      .loc[lambda df: df.index.get_level_values("Date").isin(period_)])
        
            use_box = (r >= 2) if is_fr else (c >= 2)
            (plot_pred_w_box if use_box else plot_continue_pred)(to_pt, ax, d_pt, step_x=2, obs_c=OBS_C, ms=MkS)
        
            ax.set_facecolor("ivory")
            ax.yaxis.set_major_locator(MultipleLocator(y_step))
            ax.set_ylim(ymax=y_max)
        
            last_row = (r == N_ROWS - 1)
            if last_row:
                labs = [t.get_text() for t in ax.get_xticklabels()]
                if c !=2 and not is_fr:
                    labs = [datetime.strptime(l, "%Y-%m-%d").strftime("%m\n%d") for l in labs]
                _=ax.set_xticklabels(labs)           
                ax.set_xlabel("Date")
            else:
                ax.set_xticklabels([])
                ax.set_xlabel(None)
        
            if r == 0:
                ax.set_title(FCST_Name[d_pt].upper() if not is_fr else MODEL_LABELS[c])
            if c == 0:
                label = FCST_Name[d_pt].upper() if is_fr else MODEL_LABELS[r]
                ax.set_ylabel(f"{label}\n$Q\\,(mm.day^{{-1}})$")
        
        plt.subplots_adjust(wspace=0.1, hspace=0.1)
        fg.suptitle(
            f"Discharge forecasting on {HP_X} day{'s' if HP_X != 1 else ''} lead time "
            f"from {period_[0].strftime('%b.%d')} to {period_[-1].strftime('%b.%d')} "
            f"{period_[0].year} {str_rep}", y=0.95)
        if SAVE_IMAGES:
            fg.savefig(f"{HYDRO_TO}/prev_{BENCHMARK}_bv{basin}_hp{HP_X}_"
                       f"{index_r[0].strftime('%b%d')}-{index_r[-1].strftime('%b%d')}_"
                       f"{index_r[0].year}_{CASE_CT}_wbxp.png", bbox_inches="tight")
        plt.show()

---
### END
---